# Kalenjin-English Parallel Corpus Collection

## Objective
Collect and align Kalenjin-English parallel sentence pairs for training a machine translation model.

## Motivation
Zero-shot translation via NLLB-200 proxy languages (Nuer, Luo, Swahili) **failed** — the model copies Kalenjin input rather than translating. Fine-tuning requires parallel data.

## Data Sources
1. **Bible translations** — Kalenjin Bible + English Bible aligned by verse
2. **JW300 corpus** — Jehovah's Witnesses publications (OPUS)
3. **OPUS parallel corpora** — Any existing Kalenjin entries

## Target
- Minimum 5,000 parallel sentence pairs for fine-tuning
- Ideal: 20,000-30,000 pairs from Bible verses

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import requests
import json
import re
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter

DATA_DIR = Path('../data')
RAW_DIR = DATA_DIR / 'raw'
PARALLEL_DIR = DATA_DIR / 'parallel'
RAW_DIR.mkdir(parents=True, exist_ok=True)
PARALLEL_DIR.mkdir(parents=True, exist_ok=True)

print('Directories ready')

## 2. Source 1: Bible Parallel Corpus

The Bible is the most widely translated text in the world, available in thousands of languages including Kalenjin. We use the API from bible.com / YouVersion or other open Bible APIs to fetch verse-aligned text.

### Available Kalenjin Bible Translations
- **Kalenjin Bible (Kipsigis dialect)** — available on bible.com
- **English (KJV, NIV, ESV)** — widely available

We align by book:chapter:verse to create parallel pairs.

In [ ]:
# Method 1: Try the bible-api.com (free, no auth needed)
# This API supports multiple translations

def fetch_bible_api(verse_ref, translation='kjv'):
    """Fetch a verse from bible-api.com"""
    url = f'https://bible-api.com/{verse_ref}?translation={translation}'
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            return data.get('text', '').strip()
    except Exception as e:
        pass
    return None

# Test with English
test = fetch_bible_api('John 3:16', 'kjv')
print(f'English (KJV): {test}')

In [ ]:
# Method 2: Try eBible.org which has many African language Bibles
# Check if Kalenjin is available

print('Checking eBible.org for Kalenjin translations...')
try:
    resp = requests.get('https://ebible.org/find/details.php?id=kln', timeout=10)
    if resp.status_code == 200:
        if 'Kalenjin' in resp.text or 'kln' in resp.text:
            print('  Found Kalenjin on eBible.org!')
        else:
            print('  Kalenjin page exists but may not have full text')
    else:
        print(f'  Status: {resp.status_code}')
except Exception as e:
    print(f'  Error: {e}')

# Also check for Kipsigis (a Kalenjin dialect)
for code in ['kln', 'sgc', 'niq', 'spy', 'tec']:
    try:
        resp = requests.get(f'https://ebible.org/find/details.php?id={code}', timeout=10)
        if resp.status_code == 200 and len(resp.text) > 1000:
            print(f'  {code}: Page found ({len(resp.text)} bytes)')
    except:
        pass

In [ ]:
# Method 3: Try the Digital Bible Library / Scripture Earth
# Scripture Earth has many minority language Bibles

print('Checking ScriptureEarth for Kalenjin...')
try:
    resp = requests.get('https://scriptureearth.org/data/kln/', timeout=10)
    if resp.status_code == 200:
        print(f'  Found! Page size: {len(resp.text)} bytes')
        # Look for downloadable files
        import re
        links = re.findall(r'href="([^"]+\.txt|[^"]+\.html|[^"]+\.json)"', resp.text)
        if links:
            print(f'  Downloadable files: {links[:10]}')
    else:
        print(f'  Status: {resp.status_code}')
except Exception as e:
    print(f'  Error: {e}')

# Also try find.bible API
print('\nChecking find.bible API...')
try:
    resp = requests.get('https://find.bible/bibles?language_code=kln', timeout=10)
    if resp.status_code == 200:
        print(f'  Response: {resp.text[:500]}')
except Exception as e:
    print(f'  Error: {e}')

## 3. Source 2: OPUS Parallel Corpora

OPUS (https://opus.nlpl.eu/) is the largest collection of parallel corpora. We check for any Kalenjin entries.

In [ ]:
# Check OPUS for Kalenjin
print('Checking OPUS for Kalenjin parallel data...')

# OPUS language codes for Kalenjin and dialects
opus_codes = ['kln', 'sgc', 'niq', 'spy', 'tec', 'eyo', 'pko']

for code in opus_codes:
    try:
        url = f'https://opus.nlpl.eu/opusapi/?source={code}&target=en&preprocessing=raw'
        resp = requests.get(url, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            corpora = data.get('corpora', [])
            if corpora:
                print(f'  {code}: FOUND {len(corpora)} corpora!')
                for c in corpora:
                    print(f'    - {c.get("corpus", "")} ({c.get("documents", 0)} docs, {c.get("alignment_pairs", 0)} pairs)')
            else:
                print(f'  {code}: No corpora found')
    except Exception as e:
        print(f'  {code}: Error - {e}')

In [ ]:
# Also check JW300 specifically — it covers 300+ languages
print('Checking JW300 for Kalenjin...')
try:
    url = 'https://opus.nlpl.eu/opusapi/?source=kln&target=en&preprocessing=raw&corpus=JW300'
    resp = requests.get(url, timeout=10)
    if resp.status_code == 200:
        data = resp.json()
        if data.get('corpora'):
            print(f'  JW300 Kalenjin-English: FOUND!')
            print(f'  Details: {json.dumps(data["corpora"][0], indent=2)}')
        else:
            print('  JW300: No Kalenjin entries')
except Exception as e:
    print(f'  Error: {e}')

# Check Tatoeba
print('\nChecking Tatoeba for Kalenjin...')
try:
    url = 'https://opus.nlpl.eu/opusapi/?source=kln&target=en&preprocessing=raw&corpus=Tatoeba'
    resp = requests.get(url, timeout=10)
    data = resp.json()
    if data.get('corpora'):
        print(f'  Tatoeba: FOUND!')
    else:
        print('  Tatoeba: No Kalenjin entries')
except Exception as e:
    print(f'  Error: {e}')

## 4. Source 3: Manually Create Parallel Data from Common Voice

If online sources are insufficient, we can create parallel data by translating our existing Kalenjin sentences. This section prepares the data for manual or assisted translation.

In [ ]:
# Load our Kalenjin sentences from Common Voice
cv_train = pd.read_csv('../../data/kln/train.tsv', sep='\t')
cv_test = pd.read_csv('../../data/kln/test.tsv', sep='\t')

# Get unique sentences
all_sentences = pd.concat([cv_train['sentence'], cv_test['sentence']]).dropna().unique()
print(f'Total unique Kalenjin sentences: {len(all_sentences)}')

# Analyze sentence characteristics for prioritization
sent_df = pd.DataFrame({'sentence': all_sentences})
sent_df['word_count'] = sent_df['sentence'].apply(lambda s: len(s.split()))
sent_df['char_count'] = sent_df['sentence'].apply(len)

print(f'\nWord count stats:')
print(sent_df['word_count'].describe())

# Select a diverse sample for manual translation
# Prioritize: short (easy to translate), medium, and long sentences
short = sent_df[sent_df['word_count'] <= 4].sample(min(200, len(sent_df[sent_df['word_count'] <= 4])), random_state=42)
medium = sent_df[(sent_df['word_count'] > 4) & (sent_df['word_count'] <= 8)].sample(min(500, len(sent_df[(sent_df['word_count'] > 4) & (sent_df['word_count'] <= 8)])), random_state=42)
long_s = sent_df[sent_df['word_count'] > 8].sample(min(300, len(sent_df[sent_df['word_count'] > 8])), random_state=42)

translation_candidates = pd.concat([short, medium, long_s]).reset_index(drop=True)
translation_candidates['english'] = ''  # Empty column for translations

print(f'\nSelected for translation: {len(translation_candidates)}')
print(f'  Short (1-4 words): {len(short)}')
print(f'  Medium (5-8 words): {len(medium)}')
print(f'  Long (9+ words): {len(long_s)}')

In [ ]:
# Export for manual translation
export_path = RAW_DIR / 'kalenjin_for_translation.tsv'
translation_candidates[['sentence', 'english', 'word_count']].to_csv(
    export_path, sep='\t', index=False
)
print(f'Exported {len(translation_candidates)} sentences to:')
print(f'  {export_path}')
print(f'\nInstructions:')
print(f'  1. Open the TSV file in a spreadsheet')
print(f'  2. Fill in the "english" column with translations')
print(f'  3. Save and re-import here')

# Show first 10 for preview
print(f'\nFirst 10 sentences to translate:')
for i, row in translation_candidates.head(10).iterrows():
    print(f'  {i+1}. [{row["word_count"]}w] {row["sentence"]}')

## 5. Source 4: Use LLM-Assisted Translation

As an alternative to manual translation, we can use a large language model (e.g., GPT-4, Claude) to generate initial translations, then have native speakers verify and correct them. This is faster than fully manual translation.

In [ ]:
# Prepare prompts for LLM-assisted translation
# These can be used with ChatGPT, Claude, or any LLM

prompt_template = """You are a translator specializing in Kalenjin, a Nilotic language spoken in Kenya's Rift Valley.

Translate the following Kalenjin sentences to English. If you are unsure, provide your best guess and mark it with [?].

Context: These are everyday sentences from Mozilla Common Voice recordings by Kalenjin speakers.

Kalenjin sentences:
{sentences}

Please provide translations in the format:
1. [Kalenjin] -> [English translation]
2. [Kalenjin] -> [English translation]
..."""

# Create batches of 20 sentences for LLM translation
batch_size = 20
prompts = []
for i in range(0, min(200, len(translation_candidates)), batch_size):
    batch = translation_candidates.iloc[i:i+batch_size]
    sentences = '\n'.join([f'{j+1}. {row["sentence"]}' for j, (_, row) in enumerate(batch.iterrows())])
    prompt = prompt_template.format(sentences=sentences)
    prompts.append(prompt)

# Save prompts
prompts_path = RAW_DIR / 'translation_prompts.txt'
with open(prompts_path, 'w') as f:
    for i, p in enumerate(prompts):
        f.write(f'=== BATCH {i+1} ===\n')
        f.write(p)
        f.write('\n\n')

print(f'Created {len(prompts)} translation prompts')
print(f'Saved to: {prompts_path}')
print(f'\nFirst prompt preview:')
print(prompts[0][:500])

## 6. Load and Process Parallel Data

Once parallel data is collected (from any source above), load and process it here.

In [ ]:
def load_parallel_data(filepath):
    """Load parallel data from TSV file with 'sentence' and 'english' columns."""
    df = pd.read_csv(filepath, sep='\t')
    # Filter out empty translations
    df = df[df['english'].notna() & (df['english'].str.strip() != '')]
    print(f'Loaded {len(df)} parallel pairs from {filepath}')
    return df

def normalize_kalenjin(text):
    """Same normalization as ASR training pipeline."""
    text = text.lower()
    text = text.replace('\u2018', "'").replace('\u2019', "'").replace('`', "'")
    text = text.replace('ch', 'c').replace('kh', 'k')
    text = re.sub(r"[^a-z' ]", '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def normalize_english(text):
    """Basic English normalization."""
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    return text

# Check if any parallel data exists yet
parallel_files = list(PARALLEL_DIR.glob('*.tsv'))
if parallel_files:
    print(f'Found {len(parallel_files)} parallel data files:')
    for f in parallel_files:
        print(f'  {f.name}')
else:
    print('No parallel data files found yet.')
    print('Complete steps 2-5 above to collect data, then re-run this cell.')

## 7. Create Train/Dev/Test Splits

In [ ]:
def create_splits(df, train_ratio=0.8, dev_ratio=0.1, test_ratio=0.1, seed=42):
    """Create train/dev/test splits from parallel data."""
    assert abs(train_ratio + dev_ratio + test_ratio - 1.0) < 1e-6
    
    # Shuffle
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    
    n = len(df)
    train_end = int(n * train_ratio)
    dev_end = train_end + int(n * dev_ratio)
    
    train = df[:train_end]
    dev = df[train_end:dev_end]
    test = df[dev_end:]
    
    print(f'Splits: train={len(train)}, dev={len(dev)}, test={len(test)}')
    
    # Save
    train[['sentence', 'english']].to_csv(PARALLEL_DIR / 'train.tsv', sep='\t', index=False)
    dev[['sentence', 'english']].to_csv(PARALLEL_DIR / 'dev.tsv', sep='\t', index=False)
    test[['sentence', 'english']].to_csv(PARALLEL_DIR / 'test.tsv', sep='\t', index=False)
    
    print(f'Saved to {PARALLEL_DIR}/')
    return train, dev, test

# Run this once parallel data is available:
# df = load_parallel_data(PARALLEL_DIR / 'all_parallel.tsv')
# train, dev, test = create_splits(df)
print('Split function ready. Run after collecting parallel data.')

## 8. Corpus Statistics

In [ ]:
def corpus_stats(df):
    """Print comprehensive corpus statistics."""
    print(f'Total parallel pairs: {len(df)}')
    
    # Kalenjin stats
    kln_words = df['sentence'].apply(lambda s: len(str(s).split()))
    print(f'\nKalenjin:')
    print(f'  Avg words/sentence: {kln_words.mean():.1f}')
    print(f'  Vocabulary size: {len(set(" ".join(df["sentence"].astype(str)).lower().split()))}')
    
    # English stats
    eng_words = df['english'].apply(lambda s: len(str(s).split()))
    print(f'\nEnglish:')
    print(f'  Avg words/sentence: {eng_words.mean():.1f}')
    print(f'  Vocabulary size: {len(set(" ".join(df["english"].astype(str)).lower().split()))}')
    
    # Length ratio
    ratios = eng_words / kln_words.clip(lower=1)
    print(f'\nLength ratio (eng/kln): {ratios.mean():.2f}')

# Run after loading data:
# corpus_stats(df)
print('Stats function ready.')

## 9. Summary & Next Steps

### Data Collection Status
- [ ] Bible parallel corpus
- [ ] OPUS/JW300 corpus
- [ ] Manual/LLM-assisted translations

### Once data is collected:
1. Run Section 6 to load and normalize
2. Run Section 7 to create splits
3. Run Section 8 for statistics
4. Proceed to `03_fine_tune_mt.ipynb` for model training